# Assignment 9

### Imports

In [1]:
import os
import random
import pandas as pd
import numpy as np
import torch
import logging
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
import json
import mlflow


if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
    print("Apple GPU")
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU / AMD ROCm GPU
    print("Nvidia/AMD GPU")
else:
    device = torch.device("cpu")
    print("CPU")


# Suppress annoying warnings in output
# logging.getLogger("mlflow.utils.requirements_utils").setLevel(logging.ERROR)
# logging.getLogger("mlflow.pytorch").setLevel(logging.ERROR)

# Setup mlflow local server conection
# mlflow.set_tracking_uri("http://127.0.0.1:5000")
# mlflow.set_experiment("z_prediction")
# mlflow.enable_system_metrics_logging()

CPU


### Function for making train/validation/test - split

In [3]:
def split_csvfiles(datafolder, random_seed, training_prop, validation_prop):
    csv_files = []
    for f in os.listdir(datafolder):
        if f.endswith(".csv"):
            csv_files.append(f)
    
    random.seed(random_seed)
    random.shuffle(csv_files)

    # Train/Validation/Test with proportions 80/10/10
    train_n = int(len(csv_files) * training_prop)
    val_n = int(len(csv_files) * validation_prop)
    test_n = len(csv_files) - train_n - val_n

    # Split
    if validation_prop == 0:
        train_files = csv_files[:train_n]
        test_files = csv_files[train_n:]

        return train_files, test_files
    
    else:
        train_files = csv_files[:train_n]
        val_files = csv_files[train_n: train_n + val_n]
        test_files = csv_files[train_n + val_n:]

        return train_files, val_files, test_files

### Modified enis data load function to not rely on locally defined data_dir within function

In [4]:
def load(files, data_dir):
    dataframes = []

    for f in files:
        path = os.path.join(data_dir, f)   
        df = pd.read_csv(path)           
        dataframes.append(df)             

    combined = pd.concat(dataframes, ignore_index=True)  # combine all

    return combined

### Apply to get the dataframes

In [5]:
datafolder = "../../MainProject/Assignment9/data/kinect_good_preprocessed"
random_seed = 42

# Alt 1: 80/10/10 split
# train_files, val_files, test_files = split_csvfiles(datafolder, random_seed, 0.8, 0.1)
# val_data = load(val_files, datafolder)

# Alt 2: 90/10 since we don't need validation data if we run k-fold cv
train_files, test_files = split_csvfiles(datafolder, random_seed, 0.9, 0)

train_data = load(train_files, datafolder)
test_data = load(test_files, datafolder)

### Function for splitting input (x, y) and target (z)

In [6]:
def input_target_split(dataframe):
    input_cols = []
    target_cols = []
    for c in dataframe.columns:
        if c.endswith("_x") or c.endswith("_y"):
            input_cols.append(c)
        if c.endswith("_z"):
            target_cols.append(c)
    
    input_data = dataframe[input_cols]
    target_data = dataframe[target_cols]
    return input_data, target_data

### Apply to get X_train, Y_train e.t.c

In [7]:
x_train, y_train = input_target_split(train_data)
# x_val, y_val = input_target_split(val_data)
x_test, y_test = input_target_split(test_data)

### Functions for model-handling

In [8]:
# Enis function but it doesn't rely on a global candidates_dir variable
def save_candidate_model(model, model_name, candidates_dir):
    path = os.path.join(candidates_dir, f"{model_name}.pt")
    torch.save(model.state_dict(), path)
    print(f"Saved candidate model: {path}")
    return path


# Enis load_champion function but doesn't rely on global metadata_dir variable
def load_champion_info(metadata_dir):
    path = os.path.join(metadata_dir, "champion_info.json")

    if not os.path.exists(path):
        return None

    try:
        with open(path, "r") as f:
            return json.load(f)
    except:
        return None


# Enis save_champion model but function doesn't rely on globally defined dirs
def save_champion_model(champion_dir, metadata_dir, model, model_name, mse, mae, hyperparameters):
    model_path = os.path.join(champion_dir, "champion_model.pt")
    info_path = os.path.join(metadata_dir, "champion_info.json")

    # Save model weights
    torch.save(model.state_dict(), model_path)

    # Save metadata
    info = {
        "model_name": model_name,
        "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "mse": float(mse),
        "mae": float(mae),
        "hyperparameters": hyperparameters
    }

    with open(info_path, "w") as f:
        json.dump(info, f, indent=2)

    print("New champion model saved!")


# Enis update_champion but with local variable dirs
def update_champion(metadata_dir, champion_dir, model, model_name, mse, mae, hyperparameters):
    current = load_champion_info(metadata_dir)

    if current is None:
        print("No champion found --> saving first model")
        save_champion_model(champion_dir, metadata_dir, model, model_name, mse, mae, hyperparameters)

    elif mae < current["mae"]:
        print(f"New model is better (MSE {mse} < {current['mse']})")
        save_champion_model(champion_dir, metadata_dir, model, model_name, mse, mae, hyperparameters)

    else:
        print(f"Model NOT better (MSE {mse} ≥ {current['mse']})")

### Functions for learning

In [ ]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        #nn.init.xavier_uniform_(m.weight)  # good for Tanh
        nn.init.kaiming_uniform_(m.weight) # good for ReLU
        nn.init.zeros_(m.bias)


class ZPredictor(nn.Module):
    def __init__(self, hidden_layers: list, activation=nn.ReLU(), dropout=0.0):
        super().__init__()
        
        layers = []
        input_size = 26  # 13 joints x 2 (x, y)

        activations = {"relu": nn.ReLU(),
                       "tanh": nn.Tanh(),
                       "gelu": nn.GELU(),
                       "leaky_relu": nn.LeakyReLU()
                       }
        
        act = activations[activation]
        layers = []
        prev_size = input_size

        for hidden_size in hidden_layers:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(activation)
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            input_size = hidden_size
        
        layers.append(nn.Linear(input_size, 13))  # 13 joints z output
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

### ML

In [ ]:
# Define paths
model_dir = "../assignment9_models"
candidates_dir = os.path.join(model_dir, "candidates")
champion_dir = os.path.join(model_dir, "champion")
metadata_dir = os.path.join(model_dir, "metadata")


# Convert data to tensors
X_train = torch.tensor(x_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
# X_val = torch.tensor(x_val.values, dtype=torch.float32).to(device)
# y_val = torch.tensor(y_val.values, dtype=torch.float32).to(device)
X_test = torch.tensor(x_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)


# Define search space 
search_space = [{"model_type": "dense",
                 "hidden_layers": [1, 2, 3],
                 "activation": "relu",
                 "optimizer": "adam",
                 "lr": 0.001,
                 "batch_size": 32,
                 "epochs": 50
                 }]



